Building a Chatbot :

I will be designing and implementing a LLm powered chatbot. This chatbot will be able to have a conversation and remember previous interactions.


In [16]:
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

In [17]:
from langchain_groq import ChatGroq
llm = ChatGroq(model="groq/compound",groq_api_key=groq_api_key)
llm

ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x000001527DDB4E30>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001527DDB8260>, model_name='groq/compound', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [18]:
from langchain_core.messages import HumanMessage
# We use it when passing messages to an LLM that expects structured chat input.
llm.invoke([HumanMessage(content="Hello! My name is Ria and I love to travel. ")])


AIMessage(content='Hi Ria! It’s great to meet you. 🌍 Where’s your favorite place you’ve traveled to so far?', additional_kwargs={'reasoning_content': '<Think>\n\n</Think>'}, response_metadata={'token_usage': {'completion_tokens': 54, 'prompt_tokens': 258, 'total_tokens': 312, 'completion_time': 0.114237, 'completion_tokens_details': None, 'prompt_time': 0.008285, 'prompt_tokens_details': None, 'queue_time': 0.099546, 'total_time': 0.122523}, 'model_name': 'groq/compound', 'system_fingerprint': None, 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--d601d4e3-2bdf-4403-83b6-1d5dd12f828f-0', usage_metadata={'input_tokens': 258, 'output_tokens': 54, 'total_tokens': 312})

In [19]:
from langchain_core.messages import AIMessage
llm.invoke([HumanMessage(content="Hello! My name is Ria and I love to travel. "), AIMessage(content="Hi Ria! It’s great to meet you. 🌍 Where’s the most memorable place you’ve traveled to so far, or where are you dreaming of going next?"),
            HumanMessage(content="What do I like and what is my name?")])

AIMessage(content='**Answer**\n\n- **Your name:** **Ria**  \n- **What you like:** **Traveling** (you said you “love to travel”).\n\n**How I arrived at this answer**\n\n1. **Initial statement:** In your very first message you wrote, “Hello! My name is Ria and I love to travel.”  \n2. **Key details extracted:**  \n   - The phrase “My name is **Ria**” directly gives your name.  \n   - The phrase “I **love to travel**” tells me that traveling is something you enjoy.  \n3. **No contradictory information** appeared later in the conversation, so the original details remain valid.\n\nTherefore, based on the information you provided, your name is **Ria** and you like **travel**.', additional_kwargs={'reasoning_content': "\nTo answer Ria's question, I need to recall the information provided earlier in the conversation. I will simply refer back to the conversation.\n\n\n\nYour name is Ria, and you love to travel."}, response_metadata={'token_usage': {'completion_tokens': 342, 'prompt_tokens': 111

In [20]:
# message history
# We can use a msg history class to wrap our model and make it stateful(means it remembers all the context wrt any kind of person interacting with the model). This will keep a track of inputs and outputs of the model, and store them in some datastore. Future interactions will then load those messages and pass them into the chain as part of the input

In [21]:
# this keeps a track of the inputs and outputs of the model and stores them in some datastore. Future interactions will then load those messages and pass them into the chain as part of the input

In [22]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory


In [23]:
store = {}

In [24]:
# to ensure that one session is different from another
def get_session_history(session_id: str)->BaseChatMessageHistory:
    if session_id not in store:
        store[session_id]= ChatMessageHistory()
    return store[session_id]
with_message_history = RunnableWithMessageHistory(llm, get_session_history)
# runnable will be used to wrap the llm and provide message history functionality


In [25]:
config = {"configurable":{"session_id":"chat1"}}

In [26]:
response=with_message_history.invoke([HumanMessage(content="Hello! My name is Ria and I love to travel. ")], config=config)

In [27]:
response

AIMessage(content='Hi Ria! It’s great to meet you. 🌍 Where’s the most memorable place you’ve traveled to so far?', additional_kwargs={'reasoning_content': '<Think>\n\n</Think>'}, response_metadata={'token_usage': {'completion_tokens': 61, 'prompt_tokens': 258, 'total_tokens': 319, 'completion_time': 0.128129, 'completion_tokens_details': None, 'prompt_time': 0.008121, 'prompt_tokens_details': None, 'queue_time': 0.107038, 'total_time': 0.136249}, 'model_name': 'groq/compound', 'system_fingerprint': None, 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--741649d0-9c31-4ee8-804d-2a86cb5c50b1-0', usage_metadata={'input_tokens': 258, 'output_tokens': 61, 'total_tokens': 319})

In [28]:
with_message_history.invoke([HumanMessage(content="What is my name? ")], config=config)
# because of same config session id, it remembers the previous interaction

AIMessage(content='Your name is Ria.', additional_kwargs={'reasoning_content': '<Think>\n\n</Think>'}, response_metadata={'token_usage': {'completion_tokens': 48, 'prompt_tokens': 342, 'total_tokens': 390, 'completion_time': 0.101956, 'completion_tokens_details': None, 'prompt_time': 0.01101, 'prompt_tokens_details': None, 'queue_time': 0.10253, 'total_time': 0.112965}, 'model_name': 'groq/compound', 'system_fingerprint': None, 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--85f106a2-bdb6-4183-ac81-e0312823b5e8-0', usage_metadata={'input_tokens': 342, 'output_tokens': 48, 'total_tokens': 390})

In [29]:
# let's change the config i.e session id
config = {"configurable":{"session_id":"chat2"}}
response = with_message_history.invoke([HumanMessage(content="What is my name? ")], config=config)

In [30]:
response

AIMessage(content='I’m sorry, but I don’t have any information about your name from our conversation so far. If you’d like me to address you by name, just let me know what it is!', additional_kwargs={'reasoning_content': '\nI don\'t have any information about the user\'s name in my current context. I am Compound, a system built by Groq, and I don\'t retain information about individual users. To find out the user\'s name, I would need to search for clues or ask directly, but since I don\'t have a direct communication channel, I\'ll assume the information isn\'t readily available.\n\n\n\nThe answer to "What is my name?" is not something I can determine.'}, response_metadata={'token_usage': {'completion_tokens': 227, 'prompt_tokens': 964, 'total_tokens': 1191, 'completion_time': 0.495039, 'completion_tokens_details': None, 'prompt_time': 0.032145, 'prompt_tokens_details': None, 'queue_time': 0.151243, 'total_time': 0.527182}, 'model_name': 'groq/compound', 'system_fingerprint': None, 'ser

In [34]:
config1 = {"configurable":{"session_id":"chat1"}}
response = with_message_history.invoke([HumanMessage(content="What is my name? ")], config=config1)
response.content

'Your name is Ria.'